In [1]:
# ============================================================
# LEVIR-Ship + YOLO26s PRETRAINED
# Official train/val/test split
# Google Colab ONE CELL
# ============================================================

!pip install -q -U ultralytics gdown

import os
import shutil
import zipfile
from pathlib import Path

import gdown
import yaml
import torch
from ultralytics import YOLO
from google.colab import drive


# ============================================================
# 1. Google Drive
# ============================================================

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/LEVIR_Ship_YOLO26")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DIR = Path("/content/LEVIR_Ship")
DOWNLOAD_DIR = LOCAL_DIR / "downloads"
DATASET_DIR = LOCAL_DIR / "dataset"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Official LEVIR-Ship Google Drive IDs
#    from the authors' GitHub README
# ============================================================

FILES = {
    "train": "1_kjTr4mpF1g2fWhAodKZXrsZjWrCfPh5",
    "val":   "1q2KFLVYU1SbeSU4Uksj6FV9C28YmUsb1",
    "test":  "1SRq7hq7glKzzePWqtSlt0C_RVtc1pQQC",
}


# ============================================================
# 3. Download official split ZIP files
# ============================================================

for split, file_id in FILES.items():

    zip_path = DOWNLOAD_DIR / f"{split}.zip"

    if not zip_path.exists():
        print(f"\nDownloading {split}.zip ...")

        gdown.download(
            id=file_id,
            output=str(zip_path),
            quiet=False
        )
    else:
        print(f"{split}.zip already exists.")


# ============================================================
# 4. Extract
# ============================================================

EXTRACT_DIR = LOCAL_DIR / "extracted"
EXTRACT_DIR.mkdir(exist_ok=True)

for split in ["train", "val", "test"]:

    zip_path = DOWNLOAD_DIR / f"{split}.zip"
    target = EXTRACT_DIR / split

    if not target.exists():
        print(f"\nExtracting {split} ...")
        target.mkdir(parents=True)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)

    else:
        print(f"{split} already extracted.")


# ============================================================
# 5. Find image / label files automatically
#
# Authors already provide YOLO labels.
# We normalize whatever folder nesting is inside ZIP files
# into:
#
# dataset/
#   train/images
#   train/labels
#   val/images
#   val/labels
#   test/images
#   test/labels
# ============================================================

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def prepare_split(split):

    src = EXTRACT_DIR / split

    image_files = [
        p for p in src.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    label_files = [
        p for p in src.rglob("*.txt")
        if p.is_file()
    ]

    out_img = DATASET_DIR / split / "images"
    out_lbl = DATASET_DIR / split / "labels"

    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # labels indexed by stem
    # --------------------------------------------------------

    label_map = {
        p.stem: p
        for p in label_files
    }

    copied_images = 0
    copied_labels = 0
    negatives = 0

    for img in image_files:

        dst_img = out_img / img.name

        if not dst_img.exists():
            shutil.copy2(img, dst_img)

        copied_images += 1

        # ----------------------------------------------------
        # YOLO allows empty txt for negative samples.
        # If label exists -> copy.
        # If missing -> create empty txt.
        # ----------------------------------------------------

        label = label_map.get(img.stem)

        dst_label = out_lbl / f"{img.stem}.txt"

        if label is not None:

            if not dst_label.exists():
                shutil.copy2(label, dst_label)

            copied_labels += 1

            if label.stat().st_size == 0:
                negatives += 1

        else:

            dst_label.touch(exist_ok=True)
            negatives += 1

    print(
        f"{split:5s}: "
        f"images={copied_images}, "
        f"labels={copied_labels}, "
        f"negative={negatives}"
    )


print("\n===== Preparing YOLO dataset =====")

for split in ["train", "val", "test"]:
    prepare_split(split)


# ============================================================
# 6. Create dataset.yaml
# ============================================================

YAML_PATH = LOCAL_DIR / "levir_ship.yaml"

data_yaml = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "ship"
    }
}

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )

print("\nDataset YAML:")
print(YAML_PATH)
print(YAML_PATH.read_text())


# ============================================================
# 7. Basic sanity check
# ============================================================

for split in ["train", "val", "test"]:

    n_img = len(list((DATASET_DIR / split / "images").glob("*")))
    n_lbl = len(list((DATASET_DIR / split / "labels").glob("*.txt")))

    print(
        f"{split:5s} | "
        f"images={n_img:4d} | "
        f"labels={n_lbl:4d}"
    )

    assert n_img > 0, f"No images found in {split}"
    assert n_img == n_lbl, (
        f"Image/label count mismatch in {split}: "
        f"{n_img} vs {n_lbl}"
    )


# ============================================================
# 8. Device
# ============================================================

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 9. YOLO26s
#
# .pt   -> COCO PRETRAINED
# .yaml -> training from scratch
# ============================================================

model = YOLO("yolo26s.pt")

print("\nLoaded: COCO-pretrained YOLO26s")


# ============================================================
# 10. TRAIN
#
# Original LEVIR images = 512 x 512.
# imgsz=640 slightly upsamples tiny ships.
# ============================================================

results = model.train(
    data=str(YAML_PATH),

    epochs=50,
    imgsz=640,
    batch=16,

    device=DEVICE,

    workers=2,

    pretrained=True,

    patience=10,

    optimizer="auto",

    cache=False,

    project=str(PROJECT_DIR),
    name="YOLO26s_LEVIR_Ship",

    exist_ok=True,

    plots=True,
    save=True,
)


# ============================================================
# 11. Load best checkpoint
# ============================================================

BEST_MODEL = (
    PROJECT_DIR
    / "YOLO26s_LEVIR_Ship"
    / "weights"
    / "best.pt"
)

print("\nBest model:")
print(BEST_MODEL)

best_model = YOLO(str(BEST_MODEL))


# ============================================================
# 12. VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

val_result = best_model.val(
    data=str(YAML_PATH),
    split="val",
    imgsz=640,
    batch=16,
    device=DEVICE,
)


# ============================================================
# 13. INDEPENDENT TEST
# ============================================================

print("\n" + "=" * 70)
print("TEST RESULT")
print("=" * 70)

test_result = best_model.val(
    data=str(YAML_PATH),
    split="test",
    imgsz=640,
    batch=16,
    device=DEVICE,
)


# ============================================================
# 14. Print main detection metrics
# ============================================================

print("\n" + "=" * 70)
print("FINAL TEST METRICS")
print("=" * 70)

print(f"Precision : {test_result.box.mp:.4f}")
print(f"Recall    : {test_result.box.mr:.4f}")
print(f"mAP50     : {test_result.box.map50:.4f}")
print(f"mAP50-95  : {test_result.box.map:.4f}")

print("\nDONE")
print("Best model saved at:")
print(BEST_MODEL)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 799.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


MessageError: Error: credential propagation was unsuccessful